# 🛰️ SpaceNet 8: Multi-Temporal Satellite Flood Mapping & Damage Assessment
### **KL University Capstone Project `KLCAP-2026-00010`**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isuryaprakashh/Capstone-Flood-Mapping/blob/main/notebooks/SpaceNet8_Colab_Training.ipynb)
![PyTorch](https://img.shields.io/badge/PyTorch-2.x-EE4C2C?logo=pytorch)
![GPU](https://img.shields.io/badge/Target%20GPU-NVIDIA%20Tesla%20T4%20(16GB)-76B900?logo=nvidia)
![License](https://img.shields.io/badge/License-MIT-blue)

---

### 📌 Pipeline Overview
This notebook provides the complete workflow to train and evaluate the deep learning models on a **Google Colab Free / Pro T4 GPU (16 GB VRAM)**:

1. **GPU & CUDA Environment Verification** (T4 Tensor Cores check)
2. **Google Drive Integration** (Persistent storage for datasets & trained checkpoints)
3. **Fast Dependency Installation** (`rasterio`, `geopandas`, `albumentations`, `timm`, etc.)
4. **Dataset Setup & High-Speed Mask Generation** (5-channel multi-hazard binary masks)
5. **Model 1: Baseline Early-Concatenation U-Net** (`O2 Baseline`)
6. **Model 2: Novel Siamese Cross-Attention Network** (`O3 Novel Architecture`)
7. **Comparative Evaluation & Benchmark Metrics** (mIoU, F1, Precision, Recall)
8. **Visual Inspection & Qualitative Inference** (Pre/Post imagery vs. Predictions)
9. **Checkpoint Export & Google Drive Backup** (Preserve `.pth` models permanently)

> **💡 Pro-Tip for Colab T4 GPU:**
> - **Mixed Precision (AMP)** is enabled by default to utilize the T4's Turing Tensor Cores (speeds up training 2.5x and reduces VRAM usage by 50%).
> - **Batch Size = 8** with **Tile Size = 512x512** fits safely within the 16 GB VRAM limits.
> - **Num Workers = 2** is configured to match Google Colab's 2-vCPU configuration without multiprocessing bottlenecks.

## 1️⃣ Hardware & GPU Environment Verification
Make sure your Colab runtime is set to **GPU** (Runtime ➔ Change runtime type ➔ T4 GPU).

In [ ]:
# Check NVIDIA GPU details
!nvidia-smi

import torch
print("=" * 60)
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Active GPU:       {device_name} ({vram_gb:.2f} GB VRAM)")
    print(f"Tensor Cores / AMP: {'Supported (Turing Arch)' if 'T4' in device_name or 'V100' in device_name or 'A100' in device_name else 'Supported'}")
else:
    print("⚠️ WARNING: No GPU detected! Please go to Runtime -> Change runtime type -> Select T4 GPU.")
print("=" * 60)

## 2️⃣ Mount Google Drive & Setup Repository
Mount Google Drive so your trained checkpoints (`.pth`), logs, and datasets are securely saved even if your Colab session expires.

In [ ]:
import os
from pathlib import Path
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define project working paths
WORKSPACE_DIR = Path("/content/Capstone-Flood-Mapping")
DRIVE_BACKUP_DIR = Path("/content/drive/MyDrive/SpaceNet8_Checkpoints")
DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)

# Clone repository if not already present
if not WORKSPACE_DIR.exists():
    !git clone https://github.com/isuryaprakashh/Capstone-Flood-Mapping.git /content/Capstone-Flood-Mapping
else:
    print(f"Workspace already exists at {WORKSPACE_DIR}")

%cd /content/Capstone-Flood-Mapping
!git pull
print(f"Current working directory: {os.getcwd()}")

## 3️⃣ Install Geospatial & Deep Learning Dependencies
Installs `rasterio` (GeoTIFF handling), `geopandas` & `shapely` (vector polygons & CRS conversion), `albumentations` (data augmentations), `timm` (encoder architectures), and `wandb`.

In [ ]:
# Fast installation of dependencies
!pip install -q rasterio geopandas shapely albumentations timm wandb networkx scikit-image pytest

# Verify imports
import rasterio
import geopandas
import albumentations
import timm
import networkx
print("✅ All geospatial & DL libraries imported successfully!")

## 4️⃣ Dataset Setup (SpaceNet 8 / CP1-DATASET)

You can choose between two methods to load the dataset into Colab:
- **Option A (Recommended & Fastest)**: Download official SpaceNet 8 datasets directly from AWS S3 in ~1–2 minutes.
- **Option B**: Extract an existing `CP1-DATASET.zip` from your mounted Google Drive.

### 📥 Option A: Download Directly from Official SpaceNet 8 AWS S3 (1-Click)

In [ ]:
#@title Download SpaceNet 8 Datasets from AWS S3
import os
from pathlib import Path

DATASET_DIR = Path("/content/Capstone-Flood-Mapping/CP1-DATASET")
DATASET_DIR.mkdir(parents=True, exist_ok=True)

DATA_URLS = {
    "Germany_Training_Public": "https://spacenet-dataset.s3.amazonaws.com/spacenet/SN8_floods/tarballs/Germany_Training_Public.tar.gz",
    "Louisiana-East_Training_Public": "https://spacenet-dataset.s3.amazonaws.com/spacenet/SN8_floods/tarballs/Louisiana-East_Training_Public.tar.gz",
    "Louisiana-West_Test_Public": "https://spacenet-dataset.s3.amazonaws.com/spacenet/SN8_floods/tarballs/Louisiana-West_Test_Public.tar.gz",
}

# Download and extract datasets
for name, url in DATA_URLS.items():
    target_folder = DATASET_DIR / name
    if target_folder.exists() and len(list(target_folder.glob("*"))) > 0:
        print(f"✅ {name} already extracted. Skipping.")
        continue

    tar_file = DATASET_DIR / f"{name}.tar.gz"
    print(f"⬇️ Downloading {name} from AWS S3...")
    !wget -c -q --show-progress {url} -O {tar_file}
    
    print(f"📦 Extracting {name}...")
    !tar -xzf {tar_file} -C {DATASET_DIR}
    !rm -f {tar_file}
    print(f"✅ {name} extracted successfully!")

print("\n📁 Dataset Directory Structure:")
!ls -lh /content/Capstone-Flood-Mapping/CP1-DATASET

### 💾 Option B (Alternative): Load from Google Drive Zip
Run this only if you already have a `CP1-DATASET.zip` on your Google Drive and prefer to use it.

In [ ]:
#@title (Optional) Extract Dataset from Google Drive
import zipfile
drive_zip_path = "/content/drive/MyDrive/CP1-DATASET.zip" #@param {type:"string"}
local_dataset_path = "/content/Capstone-Flood-Mapping/CP1-DATASET"

if os.path.exists(drive_zip_path):
    print(f"📦 Extracting {drive_zip_path} to {local_dataset_path}...")
    os.makedirs(local_dataset_path, exist_ok=True)
    with zipfile.ZipFile(drive_zip_path, 'r') as zip_ref:
        zip_ref.extractall(local_dataset_path)
    print("✅ Dataset extracted from Drive!")
else:
    print(f"ℹ️ No zip found at {drive_zip_path}. (Skipped if Option A was used)")

## 5️⃣ Step 1: Preprocess GeoTIFFs & Generate 5-Channel Masks
This script converts SpaceNet 8 GeoJSON/CSV annotations into aligned 5-channel raster masks:
- `Channel 0`: Building Footprints
- `Channel 1`: Road Network
- `Channel 2`: Flood Extent
- `Channel 3`: Flooded Buildings
- `Channel 4`: Flooded Roads

In [ ]:
!python scripts/tile_images.py \
    --data-root ./CP1-DATASET \
    --output-dir ./processed \
    --val-region Germany_Training_Public \
    --road-buffer 4.0

print("\n✅ Preprocessing Complete! Masks & Manifests generated:")
!ls -lh ./processed/manifests

## 6️⃣ Step 2: Train Baseline U-Net Model (O2 Baseline)
Trains the early-concatenation ResNet34 U-Net model (6 input channels: 3 pre + 3 post).

In [ ]:
#@title Training Parameters for Baseline U-Net
unet_epochs = 25 #@param {type:"integer"}
unet_batch_size = 8 #@param {type:"integer"}
unet_lr = 0.001 #@param {type:"number"}
use_amp = True #@param {type:"boolean"}

!python train.py \
    --model unet \
    --data-root ./CP1-DATASET \
    --mask-dir ./processed/masks \
    --epochs {unet_epochs} \
    --batch-size {unet_batch_size} \
    --num-workers 2 \
    --lr {unet_lr} \
    {'--amp' if use_amp else '--no-amp'} \
    --checkpoint-dir ./checkpoints \
    --log-dir ./logs \
    --no-wandb

# Backup best U-Net checkpoint to Google Drive
!cp ./checkpoints/unet_net_best.pth /content/drive/MyDrive/SpaceNet8_Checkpoints/ 2>/dev/null || true
print("💾 Backed up unet_net_best.pth to Google Drive!")

## 7️⃣ Step 3: Train Novel Siamese Cross-Attention Network (O3 Novel Model)
Trains the dual-stream Siamese encoder with multi-scale bidirectional cross-attention modules and class-weighted multi-task loss.

In [ ]:
#@title Training Parameters for Siamese Fusion Network
fusion_epochs = 40 #@param {type:"integer"}
fusion_batch_size = 8 #@param {type:"integer"}
fusion_lr = 0.001 #@param {type:"number"}
fusion_patience = 12 #@param {type:"integer"}

!python train.py \
    --model fusion \
    --data-root ./CP1-DATASET \
    --mask-dir ./processed/masks \
    --epochs {fusion_epochs} \
    --batch-size {fusion_batch_size} \
    --num-workers 2 \
    --lr {fusion_lr} \
    --patience {fusion_patience} \
    {'--amp' if use_amp else '--no-amp'} \
    --checkpoint-dir ./checkpoints \
    --log-dir ./logs \
    --no-wandb

# Backup best Fusion checkpoint to Google Drive
!cp ./checkpoints/fusion_net_best.pth /content/drive/MyDrive/SpaceNet8_Checkpoints/ 2>/dev/null || true
print("💾 Backed up fusion_net_best.pth to Google Drive!")

## 8️⃣ Step 4: Comparative Evaluation with TTA & Overlap Heuristics
Evaluates both models on the holdout validation dataset using **Test-Time Augmentation (TTA)** and **SpaceNet 8 Overlap Attribution Heuristics** for peak benchmark accuracy.

In [ ]:
!python eval.py \
    --data-root ./CP1-DATASET \
    --mask-dir ./processed/masks \
    --unet-ckpt ./checkpoints/unet_net_best.pth \
    --fusion-ckpt ./checkpoints/fusion_net_best.pth \
    --split val \
    --tta \
    --heuristics \
    --output ./logs/final_eval_benchmark.json

# Backup evaluation results to Drive
!cp ./logs/final_eval_benchmark.json /content/drive/MyDrive/SpaceNet8_Checkpoints/ 2>/dev/null || true

## 9️⃣ Step 5: Visual Qualitative Inspection
Visualize model predictions side-by-side: Pre-event RGB, Post-event RGB, Ground Truth Mask, Baseline U-Net prediction, and Siamese Fusion prediction.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from data.dataset import create_dataloaders, CLASS_NAMES
from models import build_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load test dataloader
loaders = create_dataloaders(
    data_root="./CP1-DATASET",
    mask_dir="./processed/masks",
    batch_size=1,
    tile_size=512,
    num_workers=2,
)
val_loader = loaders["val"]

# Helper to load model weights
def load_trained(model_type, ckpt_path):
    if not os.path.exists(ckpt_path):
        return None
    in_ch = 6 if model_type == 'unet' else 3
    m = build_model(model_type, in_channels=in_ch, num_classes=5, pretrained=False)
    ckpt = torch.load(ckpt_path, map_location=device)
    state = ckpt.get('model_state_dict', ckpt)
    m.load_state_dict({k.replace('module.', ''): v for k, v in state.items()})
    m.to(device).eval()
    return m

unet_model = load_trained('unet', './checkpoints/unet_net_best.pth')
fusion_model = load_trained('fusion', './checkpoints/fusion_net_best.pth')

# Color map for classes: [building, road, flood, flooded_building, flooded_road]
COLOR_MAP = np.array([
    [255, 255, 255], # 0: Building (White)
    [255, 255, 0],   # 1: Road (Yellow)
    [0, 191, 255],   # 2: Flood (Sky Blue)
    [255, 140, 0],   # 3: Flooded Building (Orange)
    [255, 0, 0],     # 4: Flooded Road (Red)
])

def mask_to_rgb(mask_tensor, threshold=0.5):
    # mask_tensor: (5, H, W)
    m = (mask_tensor > threshold).cpu().numpy()
    h, w = m.shape[1], m.shape[2]
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for c in range(5):
        idx = m[c] > 0
        rgb[idx] = COLOR_MAP[c]
    return rgb

# Fetch 1 sample and plot
for batch in val_loader:
    pre_img = batch['pre_image'].to(device)
    post_img = batch['post_image'].to(device)
    gt_mask = batch['mask'][0]
    tile_id = batch['tile_id'][0]
    
    # Unnormalize pre/post RGB for plotting
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    pre_np = np.clip((pre_img[0].cpu().numpy().transpose(1, 2, 0) * std + mean), 0, 1)
    post_np = np.clip((post_img[0].cpu().numpy().transpose(1, 2, 0) * std + mean), 0, 1)
    
    # Inferences
    with torch.no_grad():
        u_pred = torch.sigmoid(unet_model(pre_img, post_img))[0] if unet_model else None
        f_pred = torch.sigmoid(fusion_model(pre_img, post_img))[0] if fusion_model else None
    
    fig, axes = plt.subplots(1, 5, figsize=(22, 5))
    axes[0].imshow(pre_np); axes[0].set_title(f'Pre-Event ({tile_id})'); axes[0].axis('off')
    axes[1].imshow(post_np); axes[1].set_title('Post-Event Satellite'); axes[1].axis('off')
    axes[2].imshow(mask_to_rgb(gt_mask)); axes[2].set_title('Ground Truth Mask'); axes[2].axis('off')
    
    if u_pred is not None:
        axes[3].imshow(mask_to_rgb(u_pred)); axes[3].set_title('U-Net Prediction'); axes[3].axis('off')
    else:
        axes[3].text(0.5, 0.5, 'U-Net Not Available', ha='center'); axes[3].axis('off')
        
    if f_pred is not None:
        axes[4].imshow(mask_to_rgb(f_pred)); axes[4].set_title('Siamese Fusion Prediction'); axes[4].axis('off')
    else:
        axes[4].text(0.5, 0.5, 'Fusion Not Available', ha='center'); axes[4].axis('off')
        
    plt.suptitle("Legend: [White: Building | Yellow: Road | SkyBlue: Flood | Orange: Flooded Building | Red: Flooded Road]", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
    break

## 🔟 Step 6: Backup & Download Weights
Export all trained `.pth` model checkpoints and evaluation logs directly to your local computer.

In [ ]:
from google.colab import files

# Zip checkpoints and logs
!zip -r /content/SpaceNet8_Trained_Models.zip ./checkpoints ./logs

# Copy to Google Drive
!cp /content/SpaceNet8_Trained_Models.zip /content/drive/MyDrive/
print("✅ Saved /content/drive/MyDrive/SpaceNet8_Trained_Models.zip to Google Drive!")

# Trigger direct browser download
# files.download('/content/SpaceNet8_Trained_Models.zip')